# GFM Book Text-to-Speech Pipeline

Converts Quarto book chapters to audio using Google Cloud TTS.

## Prerequisites

**Google Cloud:**
- Google Cloud account with Text-to-Speech API enabled
- `gcloud` CLI installed and authenticated

**System dependencies:**
- `pandoc` - for converting Quarto to plain text
- `ffmpeg` - for audio concatenation (avoids chunk boundary artifacts)

**Install ffmpeg:**
```bash
# macOS
brew install ffmpeg

# Ubuntu/Debian
sudo apt-get install ffmpeg

# Windows (with chocolatey)
choco install ffmpeg

# Conda
conda install -c conda-forge ffmpeg
```

**Cost:** ~$30/1M chars for Chirp3-HD voices. A typical chapter (~60K chars) costs ~$1.80.

## 1. Setup

In [ ]:
# Install Python dependencies
# !pip install -q google-cloud-texttospeech

# Check for system dependencies
import shutil

deps_ok = True
for cmd in ["pandoc", "ffmpeg"]:
    if shutil.which(cmd):
        print(f"✓ {cmd} found")
    else:
        print(f"✗ {cmd} NOT FOUND - please install (see instructions above)")
        deps_ok = False

if not deps_ok:
    print("\n⚠️  Missing dependencies will cause errors. Install them before proceeding.")

In [2]:
# Authenticate with GCP (run once, follow the browser prompt)
# !gcloud auth application-default login

In [ ]:
import re
import subprocess
from pathlib import Path
from google.cloud import texttospeech

# Import pronunciation guide (same directory as notebook)
try:
    from pronunciation_guide import apply_pronunciations
    PRONUNCIATIONS_AVAILABLE = True
    print("Pronunciation guide loaded.")
except ImportError:
    PRONUNCIATIONS_AVAILABLE = False
    def apply_pronunciations(text, verbose=False):
        return text
    print("Warning: pronunciation_guide.py not found, skipping term corrections.")

print("Setup complete!")

## 2. Configuration

In [ ]:
# Path to gfm-book repository
BOOK_ROOT = Path("..")  # Update this path as needed

# Chapter to convert (update for different chapters)
CHAPTER_FILE = BOOK_ROOT / "part_2" / "p2-ch05-representations.qmd"
# CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch28-clinical-risk.qmd"
# CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch29-rare-disease.qmd"

# Output directory
OUTPUT_DIR = Path(".")  # Current directory, or set to BOOK_ROOT / "audio"
OUTPUT_DIR.mkdir(exist_ok=True)

# Voice options
# Neural2/WaveNet voices ($16/1M chars)
# VOICE_NAME = "en-US-Neural2-D"  # Male, natural
# VOICE_NAME = "en-US-Neural2-F"  # Female, natural
# VOICE_NAME = "en-US-Wavenet-D"  # Male, good balance

# Chirp3-HD voices ($30/1M chars) - most natural
VOICE_NAME = "en-US-Chirp3-HD-Charon"
# Other Chirp3-HD: Achernar, Aoede, Kore, Leda, Puck, Schedar, Zephyr

# Studio voices ($160/1M chars) - highest quality
# VOICE_NAME = "en-US-Studio-O"

# Speaking rate: 0.25 to 2.0 (1.0 = normal)
SPEAKING_RATE = 1.25  # Slightly faster, good for technical content

# Audio profile optimized for listening device
# Options: headphone-class-device, handset-class-device, small-bluetooth-speaker-class-device
AUDIO_PROFILE = "headphone-class-device"

# Apply pronunciation corrections for genomics/ML terms
APPLY_PRONUNCIATIONS = True

print(f"Chapter: {CHAPTER_FILE.name}")
print(f"Voice: {VOICE_NAME}")
print(f"Rate: {SPEAKING_RATE}x")
print(f"Audio profile: {AUDIO_PROFILE}")
print(f"Pronunciations: {'enabled' if APPLY_PRONUNCIATIONS else 'disabled'}")

## 3. Preprocessing Functions

In [ ]:
def convert_qmd_to_text(qmd_path: Path) -> str:
    """Convert Quarto file to plain text using pandoc."""
    result = subprocess.run(
        ["pandoc", str(qmd_path), "-t", "plain", "--wrap=none"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Pandoc failed: {result.stderr}")
    return result.stdout


def convert_table_to_prose(table_text: str) -> str:
    """Convert a markdown/plain table to speakable prose.
    
    Transforms:
        | Test Type | Yield | Best For |
        |-----------|-------|----------|
        | Single-gene | Variable | Known variant |
    
    Into:
        For Single-gene: yield is Variable, best for Known variant.
    """
    lines = table_text.strip().split('\n')
    
    # Filter out separator lines (all dashes/pipes/spaces)
    content_lines = [l for l in lines if not re.match(r'^[\s\-\|:]+$', l)]
    
    if len(content_lines) < 2:
        return table_text  # Not enough for header + data
    
    # Parse header
    header_line = content_lines[0]
    headers = [h.strip() for h in header_line.split('|') if h.strip()]
    
    if len(headers) < 2:
        return table_text  # Not a valid table
    
    # Build prose for each data row
    prose_parts = []
    for row_line in content_lines[1:]:
        cells = [c.strip() for c in row_line.split('|') if c.strip()]
        
        if len(cells) != len(headers):
            continue  # Skip malformed rows
        
        # First cell is usually the "name" or key
        row_prose = f"For {cells[0]}: "
        
        # Remaining cells paired with headers
        details = []
        for i, (header, value) in enumerate(zip(headers[1:], cells[1:])):
            if value and value != '-':
                # Clean up the header for speech
                header_clean = header.lower().replace('_', ' ')
                details.append(f"{header_clean} is {value}")
        
        if details:
            row_prose += ", ".join(details) + "."
            prose_parts.append(row_prose)
    
    if prose_parts:
        return "\n".join(prose_parts)
    return table_text


def preprocess_for_tts(text: str) -> str:
    """Clean text for TTS consumption."""

    # Remove non-ASCII characters that cause TTS issues (musical symbols, etc.)
    # Keep basic punctuation and common accented characters
    text = re.sub(r'[^\x00-\x7F\u00C0-\u00FF\u2018\u2019\u201C\u201D\u2013\u2014]+', ' ', text)

    # Convert markdown tables to prose BEFORE other processing
    # Match tables: lines starting with | or lines of just dashes
    table_pattern = r'((?:^[|\s].*\n)+)'
    
    def table_replacer(match):
        table_text = match.group(1)
        # Check if it looks like a table (has pipes and separator line)
        if '|' in table_text and re.search(r'^[\s\-\|:]+$', table_text, re.MULTILINE):
            return convert_table_to_prose(table_text) + "\n"
        return table_text
    
    text = re.sub(table_pattern, table_replacer, text, flags=re.MULTILINE)
    
    # Also handle plain-text tables (from pandoc) with dashed separators
    # These look like: "Header1   Header2\n----- -----\nVal1   Val2"
    dash_table_pattern = r'(^.+\n)(^[\-\s]+$\n)((?:^.+\n?)+)'
    
    def dash_table_replacer(match):
        header_line = match.group(1).strip()
        data_lines = match.group(3).strip().split('\n')
        
        # Split by multiple spaces (pandoc column alignment)
        headers = re.split(r'\s{2,}', header_line)
        headers = [h.strip() for h in headers if h.strip()]
        
        if len(headers) < 2:
            return match.group(0)
        
        prose_parts = []
        for row in data_lines:
            cells = re.split(r'\s{2,}', row)
            cells = [c.strip() for c in cells if c.strip()]
            
            if len(cells) >= len(headers):
                row_prose = f"For {cells[0]}: "
                details = []
                for header, value in zip(headers[1:], cells[1:]):
                    if value and value != '-':
                        details.append(f"{header.lower()} is {value}")
                if details:
                    row_prose += ", ".join(details) + "."
                    prose_parts.append(row_prose)
        
        if prose_parts:
            return "\n".join(prose_parts) + "\n"
        return match.group(0)
    
    text = re.sub(dash_table_pattern, dash_table_replacer, text, flags=re.MULTILINE)

    # Remove display math blocks
    text = re.sub(r"\$\$.*?\$\$", " [Equation omitted] ", text, flags=re.DOTALL)

    # Remove inline math
    text = re.sub(r"\$[^$]+\$", "", text)

    # Expand cross-references
    text = re.sub(r"@sec-ch(\d+)-[\w-]+", lambda m: f"Chapter {int(m.group(1))}", text)
    text = re.sub(r"@fig-[\w-]+", "the figure", text)
    text = re.sub(r"@tbl-[\w-]+", "the table", text)
    text = re.sub(r"@eq-[\w-]+", "the equation", text)

    # Clean citations
    text = re.sub(r"\[@[\w_-]+(?:;\s*@[\w_-]+)*\]", "[citation]", text)
    text = re.sub(r"@[\w_-]+", "[citation]", text)

    # Remove callout markers but keep content
    text = re.sub(r":::\s*\{\.callout-(\w+)[^}]*\}", r"[\1]: ", text)
    text = re.sub(r":::", "", text)

    # Clean figure references
    text = re.sub(r"!\[([^\]]*)\]\([^)]+\)", r"[Figure: \1]", text)

    # Remove Quarto layout directives
    text = re.sub(r"\{#[\w-]+[^}]*\}", "", text)
    text = re.sub(r"\{layout[^}]*\}", "", text)

    # Remove HTML comments
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)

    # Remove code blocks
    text = re.sub(r"```[\s\S]*?```", " [Code block omitted] ", text)

    # Clean markdown headers
    text = re.sub(r"^#+\s+", "", text, flags=re.MULTILINE)

    # Remove markdown emphasis
    text = re.sub(r"\*\*([^*]+)\*\*", r"\1", text)
    text = re.sub(r"\*([^*]+)\*", r"\1", text)
    text = re.sub(r"__([^_]+)__", r"\1", text)
    text = re.sub(r"_([^_]+)_", r"\1", text)
    text = re.sub(r"`([^`]+)`", r"\1", text)

    # Remove markdown links
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Clean list markers
    text = re.sub(r"^\s*[-*+]\s+", "  ", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\.\s+", "  ", text, flags=re.MULTILINE)

    # Remove "Estimated reading time" lines
    text = re.sub(r"^.*Estimated reading time.*$", "", text, flags=re.MULTILINE)

    # Remove any remaining lines that are just dashes/underscores (table remnants)
    text = re.sub(r"^[\-_\s]{3,}$", "", text, flags=re.MULTILINE)

    # Add periods to lines that don't end with punctuation (before collapsing newlines)
    lines = text.split('\n')
    processed_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and not stripped.endswith(('.', '?', '!', ':', ';', ',')):
            stripped += '.'
        processed_lines.append(stripped)
    text = '\n'.join(processed_lines)

    # CRITICAL: Collapse ALL newlines to spaces for smooth TTS
    # Double newlines (paragraph breaks) cause garbled audio artifacts
    text = re.sub(r'\n+', ' ', text)
    
    # Normalize whitespace - collapse multiple spaces
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


print("Preprocessing functions defined.")

## 4. TTS Generation Functions

In [ ]:
import tempfile
import os

def split_text_into_chunks(text: str, max_bytes: int = 4500) -> list:
    """Split text into chunks that fit within API byte limits.
    
    Splits on sentence boundaries (". ") for natural speech flow.
    """
    chunks = []
    current_chunk = ""

    # Split on sentence boundaries
    sentences = text.split(". ")

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if not sentence.endswith((".", "?", "!")):
            sentence += "."

        test_chunk = current_chunk + " " + sentence if current_chunk else sentence
        if len(test_chunk.encode("utf-8")) > max_bytes:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk = test_chunk

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def generate_audio(
    text: str,
    output_path: Path,
    voice_name: str,
    speaking_rate: float,
    audio_profile: str = None,
) -> None:
    """Generate audio from text using Google Cloud TTS.
    
    Uses LINEAR16 (WAV) format for chunks to avoid MP3 concatenation artifacts,
    then converts to MP3 at the end using ffmpeg.
    """
    client = texttospeech.TextToSpeechClient()

    chunks = split_text_into_chunks(text)
    total_chars = sum(len(c) for c in chunks)
    print(f"Processing {len(chunks)} chunks ({total_chars:,} characters)...")

    voice = texttospeech.VoiceSelectionParams(language_code="en-US", name=voice_name)
    
    # Use LINEAR16 (WAV) for clean concatenation, convert to MP3 at the end
    audio_config_params = {
        "audio_encoding": texttospeech.AudioEncoding.LINEAR16,
        "sample_rate_hertz": 24000,  # Standard high-quality rate
        "speaking_rate": speaking_rate,
    }
    
    # Add audio profile if specified
    if audio_profile:
        audio_config_params["effects_profile_id"] = [audio_profile]
    
    # Pitch only supported for non-Chirp voices
    if "Chirp" not in voice_name:
        audio_config_params["pitch"] = 0.0
    
    audio_config = texttospeech.AudioConfig(**audio_config_params)

    # Create temp directory for chunk files
    with tempfile.TemporaryDirectory() as tmpdir:
        chunk_files = []
        
        for i, chunk in enumerate(chunks):
            print(f"  Chunk {i+1}/{len(chunks)}...", end=" ", flush=True)

            synthesis_input = texttospeech.SynthesisInput(text=chunk)
            response = client.synthesize_speech(
                input=synthesis_input, voice=voice, audio_config=audio_config
            )
            
            # Save chunk as WAV
            chunk_path = os.path.join(tmpdir, f"chunk_{i:04d}.wav")
            with open(chunk_path, "wb") as f:
                f.write(response.audio_content)
            chunk_files.append(chunk_path)
            print("done")

        # Concatenate WAV files and convert to MP3 using ffmpeg
        print("\nConcatenating and converting to MP3...")
        
        # Create file list for ffmpeg concat
        list_path = os.path.join(tmpdir, "files.txt")
        with open(list_path, "w") as f:
            for chunk_path in chunk_files:
                f.write(f"file '{chunk_path}'\n")
        
        # Use ffmpeg to concatenate and convert
        output_str = str(output_path)
        result = subprocess.run([
            "ffmpeg", "-y",  # Overwrite output
            "-f", "concat",
            "-safe", "0",
            "-i", list_path,
            "-codec:a", "libmp3lame",
            "-qscale:a", "2",  # High quality MP3
            output_str
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"ffmpeg error: {result.stderr}")
            raise RuntimeError("ffmpeg concatenation failed")

    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"Saved: {output_path} ({size_mb:.1f} MB)")


print("TTS functions defined.")

## 5. Run the Pipeline

In [5]:
# Step 1: Convert Quarto to plain text
print(f"Converting {CHAPTER_FILE.name}...")
raw_text = convert_qmd_to_text(CHAPTER_FILE)
print(f"  Raw text: {len(raw_text):,} characters")

Converting p7-ch29-rare-disease.qmd...
  Raw text: 59,519 characters


In [ ]:
# Step 2: Preprocess for TTS
print("Preprocessing...")
clean_text = preprocess_for_tts(raw_text)

# Apply pronunciation guide for genomics terms
if APPLY_PRONUNCIATIONS:
    print("Applying pronunciation guide...")
    clean_text = apply_pronunciations(clean_text, verbose=True)

print(f"\n  Clean text: {len(clean_text):,} characters")
print(f"  Equations omitted: {clean_text.count('[Equation omitted]')}")
print(f"  Citations: {clean_text.count('[citation]')}")

In [7]:
# Preview the cleaned text
print("=" * 60)
print("PREVIEW (first 2000 chars):")
print("=" * 60)
print(clean_text)

PREVIEW (first 2000 chars):
Rare Disease Diagnosis.

Twenty-five thousand variants. One diagnosis. Where do you start?

Chapter Overview.

Prerequisites: This chapter assumes familiarity with variant effect prediction (Section 18), uncertainty quantification (Section 24), and basic concepts of Mendelian inheritance. Readers should understand how foundation models like AlphaMissense and Enformer generate variant scores.

Learning Objectives: After completing this chapter, you will be able to:
Describe the variant prioritization funnel and identify where foundation models contribute most.
Explain how computational predictions map to ACMG-AMP evidence categories and strengths.
Analyze how family structure (trios, segregation, phasing) enhances variant interpretation.
Distinguish germline from somatic variant interpretation contexts.
Evaluate when functional validation is needed to resolve variants of uncertain significance.

A four-year-old presents with developmental delay, hypotonia, an

In [ ]:
def estimate_tts_cost(char_count: int, voice_name: str) -> float:
    """Estimate cost in USD for Google Cloud TTS."""
    # Pricing per 1M characters (as of 2025)
    # https://cloud.google.com/text-to-speech/pricing
    if "Studio" in voice_name:
        rate = 160.00  # Studio voices
    elif "Chirp" in voice_name:
        rate = 30.00   # Chirp3-HD voices
    elif "Neural2" in voice_name or "Wavenet" in voice_name:
        rate = 16.00   # Neural2/WaveNet
    else:
        rate = 4.00    # Standard voices

    return (char_count / 1_000_000) * rate


total_chars = len(clean_text)
cost = estimate_tts_cost(total_chars, VOICE_NAME)
print(f"Processing ({total_chars:,} characters, ~${cost:.2f})...")

In [ ]:
# Step 3: Generate audio
chapter_name = CHAPTER_FILE.stem.replace(".", "-")
output_file = OUTPUT_DIR / f"{chapter_name}.mp3"

print(f"Generating audio with voice: {VOICE_NAME}")
print(f"Speaking rate: {SPEAKING_RATE}x, Audio profile: {AUDIO_PROFILE}")
generate_audio(clean_text, output_file, VOICE_NAME, SPEAKING_RATE, AUDIO_PROFILE)

## 6. Batch Processing (Optional)

Generate audio for multiple chapters at once.

In [ ]:
# List available chapters
chapters = sorted(BOOK_ROOT.glob("part_*/p*-ch*.qmd"))
print(f"Found {len(chapters)} chapters:")
for i, ch in enumerate(chapters[:10]):
    print(f"  {i+1}. {ch.relative_to(BOOK_ROOT)}")
if len(chapters) > 10:
    print(f"  ... and {len(chapters) - 10} more")

In [ ]:
# Batch convert selected chapters (uncomment and modify as needed)
# WARNING: This will use API quota for each chapter

# chapters_to_convert = [
#     BOOK_ROOT / "part_2" / "p2-ch05-representations.qmd",
#     BOOK_ROOT / "part_2" / "p2-ch06-cnns.qmd",
#     BOOK_ROOT / "part_2" / "p2-ch07-attention.qmd",
# ]

# for chapter in chapters_to_convert:
#     print(f"\n{'='*60}")
#     print(f"Processing: {chapter.name}")
#     print(f"{'='*60}")
#
#     raw = convert_qmd_to_text(chapter)
#     clean = preprocess_for_tts(raw)
#     output = OUTPUT_DIR / f"{chapter.stem}.mp3"
#     generate_audio(clean, output, VOICE_NAME, SPEAKING_RATE)

## 7. Voice Comparison (Optional)

Generate samples with different voices to compare quality.

In [ ]:
# Sample text for voice comparison
sample_text = clean_text[:3000]  # First 3000 chars

voices_to_test = [
    "en-US-Neural2-D",  # Male, natural
    "en-US-Neural2-F",  # Female, natural
    # "en-US-Studio-O",   # Male, studio (highest quality)
]

# Uncomment to generate samples
# for voice in voices_to_test:
#     output = OUTPUT_DIR / f"sample_{voice}.mp3"
#     print(f"\nGenerating sample with {voice}...")
#     generate_audio(sample_text, output, voice, SPEAKING_RATE)

---

## Reference

### Voices

| Voice Type | Example | Cost/1M chars |
|------------|---------|---------------|
| Standard | en-US-Standard-D | $4 |
| Neural2/WaveNet | en-US-Neural2-D | $16 |
| **Chirp3-HD** | en-US-Chirp3-HD-Charon | **$30** |
| Studio | en-US-Studio-O | $160 |

**Chirp3-HD voices:** Achernar, Achird, Algenib, Aoede, Charon, Kore, Leda, Puck, Schedar, Zephyr, and more.

### Audio Profiles

| Profile | Optimized For |
|---------|---------------|
| `headphone-class-device` | Earbuds, headphones (recommended) |
| `handset-class-device` | Smartphones |
| `small-bluetooth-speaker-class-device` | Portable speakers |
| `large-automotive-class-device` | Car speakers |

### AudioConfig Parameters

| Parameter | Range | Notes |
|-----------|-------|-------|
| `speaking_rate` | 0.25 - 2.0 | 1.0 = normal, 1.25-1.5 recommended |
| `pitch` | -20.0 - 20.0 | Semitones (not supported for Chirp) |
| `volume_gain_db` | -96.0 - 16.0 | Max +10 dB recommended |

### Links

- [Chirp3-HD Voices](https://cloud.google.com/text-to-speech/docs/chirp3-hd)
- [Audio Profiles](https://cloud.google.com/text-to-speech/docs/audio-profiles)
- [Pricing](https://cloud.google.com/text-to-speech/pricing)
- [Python API Reference](https://cloud.google.com/python/docs/reference/texttospeech/latest)